# Phân tích write-time A-MEM: note, link và evolve

Notebook này không mô phỏng lại A-MEM bằng logic khác. Runner bọc trực tiếp `RobustAgenticMemorySystem.add_note(...)` của official robust A-MEM và ghi lại mọi lời gọi LLM.

Với mỗi turn LoCoMo, ta quan sát:

1. **Note analysis**: raw output tạo `keywords`, `context`, `tags`.
2. **Evolution decision**: `NO_EVOLUTION`, `STRENGTHEN`, `UPDATE_NEIGHBOR`, hoặc `STRENGTHEN_AND_UPDATE`.
3. **Link/strengthen**: model chọn memory index nào để nối và thay tags của note mới.
4. **Neighbor update**: model sinh context/tags mới cho từng neighbor; notebook hiển thị chính xác metadata nào bị ghi đè.
5. **Parser/fallback diagnostics**: sai format, parser repair, placeholder, link sai index, link không thuộc tập neighbor đã retrieve.

Mỗi model chạy trong một subprocess riêng để giải phóng VRAM trước khi chuyển model.

In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys

IN_COLAB = Path('/content').exists()
PROJECT_URL = 'https://github.com/lenhokhoa23/j4f.git'
OFFICIAL_AMEM_URL = 'https://github.com/WujiangXu/AgenticMemory.git'

if IN_COLAB:
    PROJECT_DIR = Path('/content/j4f')
    AMEM_CHECKOUT = Path('/content/AgenticMemory')
else:
    candidates = [Path.cwd(), Path.cwd() / 'aamem_lab_project']
    PROJECT_DIR = next((p.resolve() for p in candidates if (p / 'aamem_lab').exists()), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError('Không tìm thấy aamem_lab_project từ working directory hiện tại.')
    AMEM_CHECKOUT = PROJECT_DIR.parent / 'A-mem-main'

if IN_COLAB:
    if (PROJECT_DIR / '.git').exists():
        subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=PROJECT_DIR, check=True)
    elif not PROJECT_DIR.exists():
        subprocess.run(['git', 'clone', PROJECT_URL, str(PROJECT_DIR)], check=True)
    if (AMEM_CHECKOUT / '.git').exists():
        subprocess.run(['git', 'pull', '--ff-only'], cwd=AMEM_CHECKOUT, check=True)
    elif not AMEM_CHECKOUT.exists():
        subprocess.run(['git', 'clone', OFFICIAL_AMEM_URL, str(AMEM_CHECKOUT)], check=True)

candidate_roots = [AMEM_CHECKOUT, AMEM_CHECKOUT / 'A-mem-main']
candidate_roots.extend([p for p in AMEM_CHECKOUT.glob('*') if p.is_dir()] if AMEM_CHECKOUT.exists() else [])
AMEM_REPO = next((p.resolve() for p in candidate_roots if (p / 'memory_layer_robust.py').exists()), None)
if AMEM_REPO is None:
    raise FileNotFoundError(f'Không tìm thấy root official robust A-MEM dưới {AMEM_CHECKOUT}')
AMEM_DATASET = AMEM_REPO / 'data' / 'locomo10.json'

print('PROJECT_DIR =', PROJECT_DIR)
print('AMEM_REPO =', AMEM_REPO)
print('AMEM_DATASET =', AMEM_DATASET, 'exists =', AMEM_DATASET.exists())

## Cài dependency

Trên Colab, chạy cell này một lần. Official A-MEM cần sentence-transformers để tìm neighbor; backend `hf` cần transformers và accelerate.

In [ ]:
INSTALL_DEPS = IN_COLAB
if INSTALL_DEPS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_DIR)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(AMEM_REPO / 'requirements.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'accelerate', 'pandas', 'matplotlib'], check=True)

import pandas as pd
print('Dependencies ready.')

## Cấu hình thí nghiệm

- `MAX_TURNS=8` đủ để thấy note và các quyết định evolve đầu tiên mà không phải build 419 turns.
- Danh sách mặc định so sánh 1.5B và 7B tuần tự. Nếu GPU thiếu VRAM, đổi model 7B thành 3B.
- `PRINT_RAW_OUTPUTS=True` làm log dài nhưng cho phép đọc ngay model đã sinh gì. Prompt đầy đủ vẫn được lưu trong JSONL nhờ `--include-prompts`.

In [ ]:
MODELS = [
    'Qwen/Qwen2.5-1.5B-Instruct',
    'Qwen/Qwen2.5-7B-Instruct',
]
# Nếu T4 không đủ VRAM cho 7B: MODELS[1] = 'Qwen/Qwen2.5-3B-Instruct'

BACKEND = 'hf'
SAMPLE_IDX = 0
MAX_TURNS = 8
HF_MAX_NEW_TOKENS = 768
SEED = 42
PRINT_RAW_OUTPUTS = True
PRINT_PROMPTS_DURING_RUN = False
TRACE_OUT_DIR = PROJECT_DIR / 'runs' / 'amem_write_trace'
TRACE_OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Models:', MODELS)
print('Output:', TRACE_OUT_DIR)

## Xem dữ liệu LoCoMo sẽ được đưa vào A-MEM

Đây là các turn đầu tiên của sample, chưa phải memory note. A-MEM sẽ chuyển mỗi turn thành chuỗi `Speaker <name>says : <text>` rồi mới gọi model tạo metadata.

In [ ]:
raw_data = json.loads(AMEM_DATASET.read_text(encoding='utf-8'))
conversation = raw_data[SAMPLE_IDX]['conversation']
source_turns = []
session_keys = sorted(
    [k for k, v in conversation.items() if re.fullmatch(r'session_\d+', k) and isinstance(v, list)],
    key=lambda k: int(k.split('_')[1]),
)
for session_key in session_keys:
    session_id = int(session_key.split('_')[1])
    timestamp = conversation.get(f'{session_key}_date_time', '')
    for turn in conversation[session_key]:
        text = turn.get('text', '')
        if turn.get('blip_caption'):
            text = f"[Image: {turn['blip_caption']}] {text}"
        source_turns.append({
            'session': session_id, 'dia_id': turn.get('dia_id'),
            'timestamp': timestamp, 'speaker': turn.get('speaker'), 'text': text,
        })
display(pd.DataFrame(source_turns[:MAX_TURNS]))

## Chạy official A-MEM và thu trace

Mỗi subprocess tạo một memory system mới từ cùng các turn, do đó so sánh model là công bằng ở input và trạng thái ban đầu. Runner không dùng cache đã build bằng model khác.

In [ ]:
def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', value).strip('_')

def run_streaming(cmd, cwd, log_path):
    print('\nRunning:', ' '.join(map(str, cmd)))
    with Path(log_path).open('w', encoding='utf-8') as log_f:
        proc = subprocess.Popen(
            cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end='')
            log_f.write(line)
            log_f.flush()
        return_code = proc.wait()
    if return_code != 0:
        tail = Path(log_path).read_text(encoding='utf-8', errors='replace').splitlines()[-100:]
        print('\n'.join(tail))
        raise RuntimeError(f'Write trace failed with exit code {return_code}; log={log_path}')

TRACE_RUNS = {}
for model in MODELS:
    model_safe = safe_name(model)
    tag = f'write_trace_s{SAMPLE_IDX}_n{MAX_TURNS}_{model_safe}'
    cmd = [
        sys.executable, '-u', '-m', 'aamem_lab.amem_write_trace_runner',
        '--amem-repo', str(AMEM_REPO),
        '--dataset', str(AMEM_DATASET),
        '--backend', BACKEND,
        '--model', model,
        '--hf-max-new-tokens', str(HF_MAX_NEW_TOKENS),
        '--sample-idx', str(SAMPLE_IDX),
        '--max-turns', str(MAX_TURNS),
        '--seed', str(SEED),
        '--include-prompts',
        '--output-dir', str(TRACE_OUT_DIR),
        '--tag', tag,
    ]
    if PRINT_RAW_OUTPUTS:
        cmd.append('--print-raw-outputs')
    if PRINT_PROMPTS_DURING_RUN:
        cmd.append('--print-prompts')
    log_path = TRACE_OUT_DIR / f'{tag}.run.log'
    run_streaming(cmd, PROJECT_DIR, log_path)
    TRACE_RUNS[model] = {
        'tag': tag,
        'turns': TRACE_OUT_DIR / f'{tag}.jsonl',
        'calls': TRACE_OUT_DIR / f'{tag}_calls.jsonl',
        'summary': TRACE_OUT_DIR / f'{tag}_summary.json',
        'log': log_path,
    }

print('\nCompleted models:', list(TRACE_RUNS))

## Đọc trace và tạo bảng so sánh

Các bảng sau tách chất lượng format khỏi chất lượng nội dung. `format_ok=True` chỉ có nghĩa model tuân thủ schema text; nó vẫn có thể sinh metadata sai về ngữ nghĩa.

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

turn_rows = []
call_rows = []
summaries = []
for model, paths in TRACE_RUNS.items():
    turn_rows.extend(read_jsonl(paths['turns']))
    call_rows.extend(read_jsonl(paths['calls']))
    summaries.append(json.loads(paths['summary'].read_text(encoding='utf-8')))

flat_calls = []
for row in call_rows:
    q = row.get('quality', {})
    flat_calls.append({
        'model': row['model'], 'dia_id': row['dia_id'], 'turn_index': row['turn_index'],
        'kind': row['kind'], 'elapsed_sec': row['elapsed_sec'],
        'format_ok': bool(q.get('format_ok')),
        'marker_coverage': q.get('marker_coverage', 0.0),
        'repair_suspected': bool(q.get('parser_repair_suspected')),
        'placeholder': bool(q.get('placeholder_detected')),
        'llm_error': bool(row.get('error')), 'parse_error': bool(row.get('parse_error')),
    })
call_df = pd.DataFrame(flat_calls)
stage_summary = call_df.groupby(['model', 'kind'], as_index=False).agg(
    calls=('kind', 'size'),
    format_ok_rate=('format_ok', 'mean'),
    repair_count=('repair_suspected', 'sum'),
    placeholder_count=('placeholder', 'sum'),
    llm_errors=('llm_error', 'sum'),
    parse_errors=('parse_error', 'sum'),
    mean_sec=('elapsed_sec', 'mean'),
)
display(stage_summary.sort_values(['kind', 'model']))

## 1. Model tạo memory note như thế nào?

Mỗi dòng là note cuối cùng thực sự được A-MEM lưu, sau khi raw output đã đi qua official parser/repair.

In [ ]:
note_table = []
for row in turn_rows:
    note = row['new_memory']
    quality = row['note_quality']
    note_table.append({
        'model': row['model'], 'dia_id': row['dia_id'], 'speaker': row['speaker'],
        'keywords': note['keywords'], 'context': note['context'], 'tags': note['tags'],
        'links_final': note['links'], 'format_ok': quality.get('format_ok'),
        'repair_suspected': quality.get('parser_repair_suspected'),
        'placeholder': quality.get('placeholder_detected'),
    })
note_df = pd.DataFrame(note_table)
display(note_df)

## 2. Model quyết định evolve như thế nào?

Turn đầu không có neighbor nên A-MEM bỏ qua evolution. Các turn sau luôn retrieve tối đa 5 neighbor trước khi hỏi model.

In [ ]:
evolve_table = []
for row in turn_rows:
    evo = row['evolution']
    evolve_table.append({
        'model': row['model'], 'dia_id': row['dia_id'],
        'decision': evo['decision'],
        'link_calls': evo['link_call_count'], 'update_calls': evo['update_call_count'],
        'conditional_call_match': evo['conditional_call_match'],
        'parsed_update_blocks': evo['parsed_update_block_count'],
        'changed_existing_memories': len(row['existing_memory_changes']),
    })
evolve_df = pd.DataFrame(evolve_table)
display(evolve_df)
print('Decision pivot:')
display(evolve_df.pivot(index='dia_id', columns='model', values='decision'))

## 3. Link model sinh có hợp lệ không?

- `proposed_links`: index model sinh trong call strengthen.
- `invalid_index`: index không tồn tại ở thời điểm đó.
- `outside_retrieved`: index tồn tại nhưng không nằm trong tập neighbor được đưa cho model. Đây là dấu hiệu hallucinated link.

In [ ]:
link_table = []
for row in turn_rows:
    link = row['connections']
    link_table.append({
        'model': row['model'], 'dia_id': row['dia_id'],
        'memory_count_before': row['memory_count_before'],
        'retrieved_neighbors': link['retrieved_neighbor_indices'],
        'proposed_links': link['proposed_links'],
        'invalid_index': link['invalid_memory_indices'],
        'outside_retrieved': link['not_in_retrieved_neighbor_set'],
        'duplicates': link['duplicate_links'],
    })
link_df = pd.DataFrame(link_table)
display(link_df[link_df['proposed_links'].map(bool)])
if not link_df['proposed_links'].map(bool).any():
    print('Không model nào chọn STRENGTHEN trong subset này. Hãy tăng MAX_TURNS để quan sát link calls.')

## 4. Neighbor nào đã bị evolve/ghi đè?

Bảng này là diff state thật trước/sau `add_note`, không dựa vào lời giải thích của model.

In [ ]:
update_rows = []
for row in turn_rows:
    for change in row['existing_memory_changes']:
        update_rows.append({
            'model': row['model'], 'trigger_dia_id': row['dia_id'],
            'neighbor_memory_index': change['index'],
            'neighbor_content': change.get('content', ''),
            'changed_fields': list(change.get('fields', {})),
            'field_diff': change.get('fields', {}),
        })
update_df = pd.DataFrame(update_rows)
display(update_df if not update_df.empty else pd.DataFrame({'message': ['Không có existing memory nào bị sửa trong subset này.']}))

## 5. Đọc prompt, raw output và parser output cho một case

Dùng hàm dưới để kiểm tra một model/turn/stage cụ thể. Đây là phần quan trọng để phân biệt: model sai format, parser sửa sai, hay model đúng format nhưng sai ngữ nghĩa.

In [ ]:
def show_call(model=None, dia_id=None, kind=None, occurrence=0, show_prompt=True):
    selected = call_rows
    if model is not None:
        selected = [r for r in selected if r['model'] == model]
    if dia_id is not None:
        selected = [r for r in selected if r['dia_id'] == dia_id]
    if kind is not None:
        selected = [r for r in selected if r['kind'] == kind]
    if not selected:
        print('Không tìm thấy call phù hợp.')
        return None
    row = selected[occurrence]
    print('MODEL:', row['model'])
    print('TURN:', row['dia_id'], 'STAGE:', row['kind'])
    print('QUALITY:', json.dumps(row.get('quality', {}), ensure_ascii=False, indent=2))
    if show_prompt:
        print('\n=== PROMPT ===\n', row.get('prompt', ''))
    print('\n=== RAW OUTPUT ===\n', row.get('raw_output', ''))
    print('\n=== PARSED OUTPUT ===\n', json.dumps(row.get('parsed_output'), ensure_ascii=False, indent=2))
    return row

# Ví dụ: note analysis của cùng turn đầu tiên trên từng model.
for model in MODELS:
    print('\n' + '=' * 110)
    show_call(model=model, dia_id=source_turns[0]['dia_id'], kind='note_analysis', show_prompt=False)

## 6. Tự động in một ví dụ cho từng stage và từng model

Stage nào không xuất hiện nghĩa là decision trước đó không kích hoạt call điều kiện, không phải runner bỏ sót.

In [ ]:
for model in MODELS:
    print('\n' + '#' * 110)
    print('MODEL:', model)
    for kind in ['note_analysis', 'evolve_decision', 'link_strengthen', 'neighbor_update']:
        candidates = [r for r in call_rows if r['model'] == model and r['kind'] == kind]
        if not candidates:
            print(f'\n[{kind}] không xuất hiện trong {MAX_TURNS} turns.')
            continue
        print('\n' + '-' * 110)
        show_call(model=model, dia_id=candidates[0]['dia_id'], kind=kind, show_prompt=False)

## Cách kết luận từ notebook

Không kết luận model tốt chỉ vì `format_ok_rate` cao. Cần đọc đồng thời bốn lớp:

1. **Format**: model có theo đúng section markers không?
2. **Parse**: official parser biến raw output thành object nào, có âm thầm repair/default không?
3. **Structural validity**: link có tồn tại và thuộc retrieved neighbor set không; update có đủ block không?
4. **Semantic validity**: context/tags/link/update có thực sự đúng với nội dung LoCoMo không? Phần này hiện cần người đọc hoặc một evaluator riêng; official robust A-MEM chưa có semantic validator.

Đặc biệt, fallback chỉ cứu lỗi format/crash. Nếu model sinh output đúng format nhưng nội dung sai, A-MEM vẫn lưu và có thể propagate lỗi qua links/evolution.